# Algorithms 20: Hodgkin-Huxley Model of Action Potentials**Learning Objectives:**1. Translate massive systems of biological ODEs into Python code2. Handle time-dependent stimulus functions $s(t)$3. Simulate a neuron firing an Action Potential (Spike)**Prerequisites:** Algorithms 18**Estimated time:** 120 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapter 20

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "20_hodgkin_huxley",    "level": 2,    "title": "Algorithms 20: Hodgkin-Huxley Model of Action Potentials",    "prerequisites": [        "Algorithms 18"    ],    "skills_taught": [        "lesson.algorithms_20_hodgkin_huxley"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "21_hash_tables.ipynb"}SKILLS = [    {        "id": "lesson.algorithms_20_hodgkin_huxley",        "title": "Lesson Algorithms 20 Hodgkin Huxley",        "notebook": "20_hodgkin_huxley.ipynb",        "level": 2    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "20_hodgkin_huxley.ipynb",        "level": 2    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Algorithms 18.

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.algorithms_20_hodgkin_huxley', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setupimport mathimport matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### The Biophysics of a Neuron> 📖 *From Algorithms Ch. 20:* In 1952, Alan Hodgkin and Andrew Huxley published a differential equation model of “action potentials” in the voltage of neurons, winning a Nobel Prize.The current $I$ entering a neuron consists of:- $s(t)$: External stimulus (like an electrode)- $I_{Na}$: Sodium channel flux (increases voltage)- $I_{K}$: Potassium channel flux (decreases voltage)- $I_{L}$: Leakage currentThe voltage equation is: $\frac{dV}{dt} = \frac{1}{C}[s(t) - I_{Na} - I_{K} - I_{L}]$The channel currents depend on gating variables $n$ (potassium activation), $m$ (sodium activation), and $h$ (sodium inactivation), which themselves are governed by their own differential equations:$\frac{dn}{dt} = \alpha_n(V)(1 - n) - \beta_n(V)n$The $\alpha$ and $\beta$ functions are complex exponentials derived experimentally.This is a 4-variable nonlinear system: State is $(V, n, m, h)$!

### Comprehension Check ✅1. Why are $n$, $m$, and $h$ restricted between 0 and 1?2. Notice that the external stimulus is a function of time, $s(t)$. If $s(t) = 0$, what should the neuron's voltage do?<details><summary>💡 Explanation after a retrieval attempt</summary>1. They represent the *probability* that a certain ion channel gate is open. $0 = 0\%$ open, $1 = 100\%$ open.2. It should stay at its resting potential ($V=0$ in our shifted coordinate system). If bumped slightly, the leakage current $I_L$ will drag it back to 0.</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: The $\alpha$ and $\beta$ FunctionsLet's write out the mathematical functions for the Potassium ($n$) gate.$\alpha_n(V) = 0.01(10 - V) / (\exp(0.1(10 - V)) - 1)$$\beta_n(V) = 0.125 \exp(-V / 80)$*(Careful! If $V=10$, $\alpha_n$ divides by zero! Use `V = 10.0001` or L'Hopital's limit in your code)*

In [ ]:
import mathdef alpha_n(V):    if abs(10 - V) < 1e-6: V += 1e-6 # prevent division by zero    return 0.01 * (10 - V) / (math.exp(0.1 * (10 - V)) - 1)def beta_n(V):    return 0.125 * math.exp(-V / 80)# print(alpha_n(0), beta_n(0))

---## Phase 1 — Algorithm Construction### Step 1: Implementing the rest of the constantsWrite the $\alpha$ and $\beta$ functions for $m$ and $h$, and define the constants.$\alpha_m(V) = 0.1(25 - V) / (\exp(0.1(25 - V)) - 1)$$\beta_m(V) = 4 \exp(-V / 18)$$\alpha_h(V) = 0.07 \exp(-V / 20)$$\beta_h(V) = 1 / (\exp(0.1(30 - V)) + 1)$

In [ ]:
# CONSTANTSC = 1.0V_Na = 115V_K = -12V_L = 10.6g_Na_bar = 120g_K_bar = 36g_L_bar = 0.3# YOUR CODE HERE to define alpha_m, beta_m, alpha_h, beta_hpass

### Step 2: The Time-Dependent StimulusWe will inject current pulses at specific time intervals to trigger spikes. Write $s(t)$ which returns 150 if $t$ is inside certain intervals, and 0 otherwise.The intervals are: `[10, 11], [20, 21], [30, 40], [50, 51], [53, 54], [56, 57], [59, 60], [62, 63], [65, 66]`.

In [ ]:
def s_t(t):    intervals = [(10,11), (20,21), (30,40), (50,51), (53,54), (56,57), (59,60), (62,63), (65,66)]    for start, end in intervals:        if start <= t <= end:            return 150.0    return 0.0

### Step 3: The 4 Differential EquationsState is `(V, n, m, h)`. Write `dV_dt`, `dn_dt`, `dm_dt`, `dh_dt`.Recall:$I_{Na} = g_{Na} \cdot m^3 h \cdot (V - V_{Na})$$I_{K} = g_{K} \cdot n^4 \cdot (V - V_{K})$$I_{L} = g_{L} \cdot (V - V_{L})$$dV/dt = (s(t) - I_{Na} - I_{K} - I_{L}) / C$

In [ ]:
# def dV_dt(t, state):#     V, n, m, h = state#     I_Na = g_Na_bar * (m**3) * h * (V - V_Na)#     I_K = g_K_bar * (n**4) * (V - V_K)#     I_L = g_L_bar * (V - V_L)#     return (s_t(t) - I_Na - I_K - I_L) / C# def dn_dt(t, state):#     V, n, m, h = state#     return alpha_n(V)*(1 - n) - beta_n(V)*n# WRITE dm_dt and dh_dt# YOUR CODE HERE

---## Phase 2 — Experimental Verification### Firing the NeuronBring over `MultiEulerEstimator`. Initial state at $t=0$ is $V = 0$, and the asymptotic limits for $n, m, h$: $n_0 = \alpha_n(0) / (\alpha_n(0) + \beta_n(0))$Step from 0 to 80 ms with $\Delta t = 0.01$ ms. Plot $V(t)$.

In [ ]:
# PASTE MultiEulerEstimator HERE# Calculate initial state# n0 = alpha_n(0) / (alpha_n(0) + beta_n(0))# m0 = alpha_m(0) / (alpha_m(0) + beta_m(0))# h0 = alpha_h(0) / (alpha_h(0) + beta_h(0))# initial_state = (0.0, n0, m0, h0)# multi_est = MultiEulerEstimator([dV_dt, dn_dt, dm_dt, dh_dt])# pts = multi_est.calc_estimated_points((0.0, initial_state), 0.01, 80.0)# ts = [p[0] for p in pts]# Vs = [p[1][0] for p in pts]# plt.plot(ts, Vs, color='black')# plt.title("Hodgkin-Huxley Action Potentials")# plt.xlabel("Time (ms)"); plt.ylabel("Voltage (mV)"); plt.grid(True)# plt.show()

### 🔍 ReflectionNotice the long stimulus from $t=30$ to $40$ causes a train of multiple rapid spikes! This is how your nervous system encodes high-intensity stimuli: not by larger spikes (spikes are all-or-nothing), but by a higher *frequency* of spikes.---📚 **Next:** Algorithms 21 (Hash Tables)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.algorithms_20_hodgkin_huxley', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='21_hash_tables.ipynb')